In [1]:
!pip install -q gradio transformers accelerate torch

In [2]:
import gradio as gr
import re
from datetime import datetime
from typing import Tuple, Dict, List

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print("⏳ Loading model... (5-10 mins first time)")

tokenizer = AutoTokenizer.from_pretrained(
    "ibm-granite/granite-3.3-2b-instruct"
)
model = AutoModelForCausalLM.from_pretrained(
    "ibm-granite/granite-3.3-2b-instruct"
)

print("✅ Model Ready!")

⏳ Loading model... (5-10 mins first time)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Model Ready!


In [4]:
class ColorPaletteGenerator:

    @staticmethod
    def generate_palette(style: str = "modern") -> Dict[str, str]:

        palettes = {
            'modern': {
                'primary':        '#667eea',
                'secondary':      '#764ba2',
                'accent':         '#f093fb',
                'background':     '#0f172a',
                'surface':        '#1e293b',
                'text_primary':   '#f1f5f9',
                'text_secondary': '#cbd5e1',
                'border':         '#334155'
            },
            'vibrant': {
                'primary':        '#ff6b6b',
                'secondary':      '#ff8e72',
                'accent':         '#ff5566',
                'background':     '#0f172a',
                'surface':        '#1e293b',
                'text_primary':   '#f1f5f9',
                'text_secondary': '#cbd5e1',
                'border':         '#334155'
            },
            'ocean': {
                'primary':        '#0ea5e9',
                'secondary':      '#38bdf8',
                'accent':         '#7dd3fc',
                'background':     '#0f172a',
                'surface':        '#1e293b',
                'text_primary':   '#f1f5f9',
                'text_secondary': '#cbd5e1',
                'border':         '#334155'
            },
            'forest': {
                'primary':        '#22c55e',
                'secondary':      '#4ade80',
                'accent':         '#86efac',
                'background':     '#0f172a',
                'surface':        '#1e293b',
                'text_primary':   '#f1f5f9',
                'text_secondary': '#cbd5e1',
                'border':         '#334155'
            },
            'sunset': {
                'primary':        '#f97316',
                'secondary':      '#fb923c',
                'accent':         '#fdba74',
                'background':     '#0f172a',
                'surface':        '#1e293b',
                'text_primary':   '#f1f5f9',
                'text_secondary': '#cbd5e1',
                'border':         '#334155'
            },
            'luxury': {
                'primary':        '#eab308',
                'secondary':      '#facc15',
                'accent':         '#fde047',
                'background':     '#0f172a',
                'surface':        '#1e293b',
                'text_primary':   '#f1f5f9',
                'text_secondary': '#cbd5e1',
                'border':         '#334155'
            },
            'creative': {
                'primary':        '#9b59b6',
                'secondary':      '#8e44ad',
                'accent':         '#e74c3c',
                'background':     '#0f172a',
                'surface':        '#1e293b',
                'text_primary':   '#f1f5f9',
                'text_secondary': '#cbd5e1',
                'border':         '#334155'
            },
            'minimal': {
                'primary':        '#2c3e50',
                'secondary':      '#34495e',
                'accent':         '#3498db',
                'background':     '#0f172a',
                'surface':        '#1e293b',
                'text_primary':   '#f1f5f9',
                'text_secondary': '#cbd5e1',
                'border':         '#334155'
            }
        }

        return palettes.get(style.lower(), palettes['modern'])

print("✅ ColorPaletteGenerator Ready!")

✅ ColorPaletteGenerator Ready!


In [5]:
class HTMLPageBuilder:

    @staticmethod
    def extract_sections(description: str) -> List[str]:
        sections = []

        section_keywords = {
            'hero':         ['hero', 'banner', 'landing'],
            'features':     ['features', 'services', 'benefits'],
            'about':        ['about', 'team', 'company'],
            'portfolio':    ['portfolio', 'gallery', 'work', 'projects'],
            'testimonials': ['testimonial', 'reviews', 'feedback'],
            'pricing':      ['pricing', 'plans', 'cost'],
            'contact':      ['contact', 'form', 'reach'],
            'footer':       ['footer', 'bottom']
        }

        desc_lower = description.lower()

        for section_type, keywords in section_keywords.items():
            for keyword in keywords:
                if keyword in desc_lower:
                    if section_type not in sections:
                        sections.append(section_type)
                    break

        if 'hero' not in sections:
            sections.insert(0, 'hero')
        if 'footer' not in sections:
            sections.append('footer')

        return sections

    @staticmethod
    def generate_css(palette: Dict) -> str:
        p = palette
        return f"""<style>
        * {{ margin:0; padding:0; box-sizing:border-box; }}
        html {{ scroll-behavior:smooth; }}
        body {{
            font-family: 'Segoe UI', sans-serif;
            background: {p['background']};
            color: {p['text_primary']};
        }}
        a {{ text-decoration:none; transition: all 0.3s; }}
        input, textarea {{
            width: 100%;
            padding: 14px;
            border-radius: 10px;
            border: 1px solid {p['border']};
            background: {p['background']};
            color: {p['text_primary']};
            font-size: 1rem;
            margin-bottom: 20px;
        }}
        @media (max-width: 768px) {{
            h1 {{ font-size: 2rem !important; }}
            nav div {{ display: none; }}
            .grid {{ grid-template-columns: 1fr !important; }}
        }}
        </style>"""

    @staticmethod
    def generate_section_html(section_type: str,
                               palette: Dict) -> str:
        p = palette
        year = datetime.now().year

        sections = {

            'hero': f"""
<section id="hero" style="min-height:100vh;display:flex;
  align-items:center;justify-content:center;text-align:center;
  background:linear-gradient(135deg,{p['primary']},{p['secondary']});
  padding:80px 20px;">
  <div>
    <h1 style="font-size:3.5rem;color:#fff;
      margin-bottom:20px;font-weight:800;
      animation:fadeIn 1s ease;">
      Welcome to Your Site
    </h1>
    <p style="font-size:1.3rem;color:rgba(255,255,255,0.85);
      max-width:600px;margin:0 auto 40px;line-height:1.7;">
      Create amazing web experiences with AI-powered design
    </p>
    <a href="#features"
      style="padding:16px 40px;background:{p['accent']};
      color:#fff;border-radius:50px;font-weight:700;
      font-size:1.1rem;box-shadow:0 8px 30px rgba(0,0,0,0.3);">
      Get Started →
    </a>
  </div>
</section>""",

            'features': f"""
<section id="features" style="padding:80px 40px;
  background:{p['background']};">
  <h2 style="text-align:center;color:{p['primary']};
    font-size:2.5rem;margin-bottom:16px;font-weight:800;">
    Our Features
  </h2>
  <p style="text-align:center;color:{p['text_secondary']};
    margin-bottom:48px;font-size:1.1rem;">
    Everything you need to succeed
  </p>
  <div class="grid" style="display:grid;
    grid-template-columns:repeat(auto-fit,minmax(250px,1fr));
    gap:32px;max-width:1100px;margin:0 auto;">
    <div style="background:{p['surface']};border-radius:16px;
      padding:32px;border:1px solid {p['border']};text-align:center;">
      <div style="font-size:2.5rem;margin-bottom:16px;">⚡</div>
      <h3 style="color:{p['primary']};margin-bottom:12px;
        font-size:1.3rem;">Fast Performance</h3>
      <p style="color:{p['text_secondary']};line-height:1.6;">
        Lightning speed results every time</p>
    </div>
    <div style="background:{p['surface']};border-radius:16px;
      padding:32px;border:1px solid {p['border']};text-align:center;">
      <div style="font-size:2.5rem;margin-bottom:16px;">🎨</div>
      <h3 style="color:{p['primary']};margin-bottom:12px;
        font-size:1.3rem;">Beautiful Design</h3>
      <p style="color:{p['text_secondary']};line-height:1.6;">
        Stunning visuals that impress</p>
    </div>
    <div style="background:{p['surface']};border-radius:16px;
      padding:32px;border:1px solid {p['border']};text-align:center;">
      <div style="font-size:2.5rem;margin-bottom:16px;">🔒</div>
      <h3 style="color:{p['primary']};margin-bottom:12px;
        font-size:1.3rem;">Secure & Safe</h3>
      <p style="color:{p['text_secondary']};line-height:1.6;">
        Enterprise-grade security built in</p>
    </div>
  </div>
</section>""",

            'about': f"""
<section id="about" style="padding:80px 40px;
  background:{p['surface']};">
  <div style="max-width:1000px;margin:0 auto;display:grid;
    grid-template-columns:1fr 1fr;gap:60px;align-items:center;">
    <div>
      <h2 style="color:{p['primary']};font-size:2.5rem;
        margin-bottom:20px;font-weight:800;">About Us</h2>
      <p style="color:{p['text_secondary']};line-height:1.8;
        font-size:1.1rem;margin-bottom:16px;">
        We are passionate about creating exceptional digital
        experiences that make a real difference in people's lives.
      </p>
      <p style="color:{p['text_secondary']};line-height:1.8;">
        Our expert team combines creativity with cutting-edge
        technology to deliver outstanding results.
      </p>
    </div>
    <div style="background:{p['background']};border-radius:20px;
      height:300px;display:flex;align-items:center;
      justify-content:center;font-size:5rem;
      border:1px solid {p['border']};">🏢</div>
  </div>
</section>""",

            'portfolio': f"""
<section id="portfolio" style="padding:80px 40px;
  background:{p['background']};">
  <h2 style="text-align:center;color:{p['primary']};
    font-size:2.5rem;margin-bottom:48px;font-weight:800;">
    Our Work
  </h2>
  <div class="grid" style="display:grid;
    grid-template-columns:repeat(auto-fit,minmax(280px,1fr));
    gap:28px;max-width:1100px;margin:0 auto;">
    {"".join([f'''<div style="background:{p['surface']};
      border-radius:16px;overflow:hidden;
      border:1px solid {p['border']};">
      <div style="height:180px;background:linear-gradient(
        135deg,{p['primary']},{p['secondary']});
        display:flex;align-items:center;justify-content:center;
        font-size:3rem;">{icon}</div>
      <div style="padding:24px;">
        <h3 style="color:{p['primary']};margin-bottom:8px;">
          {name}</h3>
        <p style="color:{p['text_secondary']};font-size:.9rem;">
          {desc}</p>
      </div>
    </div>'''
    for icon,name,desc in [
      ('🌐','Web Platform','Modern web application'),
      ('📱','Mobile App','Cross-platform mobile'),
      ('🛒','E-Commerce','Online shopping platform'),
      ('📊','Dashboard','Analytics dashboard'),
      ('🎨','Brand Design','Visual identity system'),
      ('🤖','AI Tool','Machine learning product')
    ]])}
  </div>
</section>""",

            'testimonials': f"""
<section id="testimonials" style="padding:80px 40px;
  background:{p['surface']};">
  <h2 style="text-align:center;color:{p['primary']};
    font-size:2.5rem;margin-bottom:48px;font-weight:800;">
    What Clients Say
  </h2>
  <div class="grid" style="display:grid;
    grid-template-columns:repeat(auto-fit,minmax(280px,1fr));
    gap:32px;max-width:1000px;margin:0 auto;">
    {"".join([f'''<div style="background:{p['background']};
      border-radius:16px;padding:32px;
      border:1px solid {p['border']};">
      <div style="color:{p['accent']};font-size:2rem;
        margin-bottom:16px;">★★★★★</div>
      <p style="color:{p['text_secondary']};line-height:1.7;
        margin-bottom:20px;font-style:italic;">"{quote}"</p>
      <div style="display:flex;align-items:center;gap:12px;">
        <div style="width:44px;height:44px;border-radius:50%;
          background:linear-gradient(135deg,{p['primary']},
          {p['secondary']});display:flex;align-items:center;
          justify-content:center;font-size:1.2rem;">{av}</div>
        <div>
          <div style="color:{p['text_primary']};font-weight:700;">
            {name}</div>
          <div style="color:{p['text_secondary']};font-size:.85rem;">
            {role}</div>
        </div>
      </div>
    </div>'''
    for av,name,role,quote in [
      ('👩','Sarah Johnson','CEO, TechCorp',
       'Outstanding work! Exceeded all expectations.'),
      ('👨','Mike Chen','Designer',
       'The best tool I have used. Highly recommend!'),
      ('👩','Priya Sharma','Developer',
       'Saved us weeks of development time.')
    ]])}
  </div>
</section>""",

            'pricing': f"""
<section id="pricing" style="padding:80px 40px;
  background:{p['background']};">
  <h2 style="text-align:center;color:{p['primary']};
    font-size:2.5rem;margin-bottom:16px;font-weight:800;">
    Pricing Plans
  </h2>
  <p style="text-align:center;color:{p['text_secondary']};
    margin-bottom:48px;">Choose the plan that works for you</p>
  <div class="grid" style="display:grid;
    grid-template-columns:repeat(auto-fit,minmax(260px,1fr));
    gap:28px;max-width:900px;margin:0 auto;">
    {"".join([f'''<div style="background:{p['surface']};
      border-radius:20px;padding:36px;text-align:center;
      border:{('3px solid '+p['primary']) if highlight else
              ('1px solid '+p['border'])};
      {'transform:scale(1.05);' if highlight else ''}">
      {'<div style="background:'+p['primary']+';color:#fff;padding:6px 16px;border-radius:20px;font-size:.8rem;margin-bottom:16px;display:inline-block;">POPULAR</div>' if highlight else ''}
      <h3 style="color:{p['text_primary']};font-size:1.4rem;
        margin-bottom:12px;">{plan}</h3>
      <div style="font-size:2.8rem;font-weight:800;
        color:{p['primary']};margin-bottom:8px;">{price}</div>
      <div style="color:{p['text_secondary']};
        margin-bottom:24px;">per month</div>
      <ul style="list-style:none;margin-bottom:32px;text-align:left;">
        {"".join([f'<li style="padding:8px 0;color:{p[chr(116)+chr(101)+chr(120)+chr(116)+"_secondary"]};border-bottom:1px solid {p["border"]};">✓ {f}</li>' for f in features])}
      </ul>
      <a href="#contact" style="display:block;padding:14px;
        background:{'linear-gradient(135deg,'+p['primary']+','+p['secondary']+')' if highlight else p['surface']};
        color:{'#fff' if highlight else p['primary']};
        border-radius:10px;font-weight:700;
        border:1px solid {p['primary']};">
        Get Started
      </a>
    </div>'''
    for plan,price,features,highlight in [
      ('Basic','$9',['5 Projects','10GB Storage','Email Support'],False),
      ('Pro','$29',['Unlimited Projects','100GB Storage','Priority Support','Analytics'],True),
      ('Enterprise','$99',['Everything in Pro','Custom Domain','Dedicated Support','SLA'],False)
    ]])}
  </div>
</section>""",

            'contact': f"""
<section id="contact" style="padding:80px 40px;
  background:{p['surface']};">
  <h2 style="text-align:center;color:{p['primary']};
    font-size:2.5rem;margin-bottom:48px;font-weight:800;">
    Contact Us
  </h2>
  <div style="max-width:600px;margin:0 auto;
    background:{p['background']};border-radius:24px;
    padding:48px;border:1px solid {p['border']};">
    <div style="margin-bottom:20px;">
      <label style="color:{p['text_primary']};display:block;
        margin-bottom:8px;font-weight:600;">Your Name</label>
      <input type="text" placeholder="John Doe"
        style="width:100%;padding:14px;border-radius:10px;
        border:1px solid {p['border']};
        background:{p['surface']};color:{p['text_primary']};">
    </div>
    <div style="margin-bottom:20px;">
      <label style="color:{p['text_primary']};display:block;
        margin-bottom:8px;font-weight:600;">Email Address</label>
      <input type="email" placeholder="john@example.com"
        style="width:100%;padding:14px;border-radius:10px;
        border:1px solid {p['border']};
        background:{p['surface']};color:{p['text_primary']};">
    </div>
    <div style="margin-bottom:32px;">
      <label style="color:{p['text_primary']};display:block;
        margin-bottom:8px;font-weight:600;">Message</label>
      <textarea rows="5" placeholder="Tell us about your project..."
        style="width:100%;padding:14px;border-radius:10px;
        border:1px solid {p['border']};
        background:{p['surface']};color:{p['text_primary']};
        resize:vertical;"></textarea>
    </div>
    <button style="width:100%;padding:16px;
      background:linear-gradient(135deg,{p['primary']},
      {p['secondary']});color:#fff;border:none;
      border-radius:12px;font-size:1.1rem;
      font-weight:700;cursor:pointer;
      box-shadow:0 8px 30px rgba(0,0,0,0.3);">
      Send Message ✉️
    </button>
  </div>
</section>""",

            'footer': f"""
<footer id="footer" style="background:{p['surface']};
  border-top:1px solid {p['border']};padding:48px 40px;">
  <div style="max-width:1100px;margin:0 auto;display:flex;
    justify-content:space-between;align-items:center;
    flex-wrap:wrap;gap:20px;">
    <div>
      <div style="color:{p['primary']};font-size:1.4rem;
        font-weight:800;margin-bottom:8px;">Your Brand</div>
      <div style="color:{p['text_secondary']};font-size:.9rem;">
        Building amazing digital experiences
      </div>
    </div>
    <div style="display:flex;gap:20px;">
      {"".join([f'<a href="#" style="color:{p["text_secondary"]};font-size:1.5rem;">{icon}</a>'
        for icon in ['🐦','💼','📸','▶️']])}
    </div>
    <div style="color:{p['text_secondary']};font-size:.85rem;">
      &copy; {year} Your Brand. All rights reserved.
    </div>
  </div>
</footer>"""
        }

        return sections.get(section_type, '')

print("✅ HTMLPageBuilder Ready!")

✅ HTMLPageBuilder Ready!


In [6]:
class HTMLGenerationEngine:

    @staticmethod
    def generate_page(title: str,
                      description: str,
                      style: str = "modern") -> Tuple[str, str]:

        palette  = ColorPaletteGenerator.generate_palette(style)
        sections = HTMLPageBuilder.extract_sections(description)
        css      = HTMLPageBuilder.generate_css(palette)

        nav_links = ' '.join([
            f'<a href="#{s}" style="color:{palette["text_secondary"]};'
            f'margin-left:24px;font-weight:500;'
            f'transition:color 0.3s;"'
            f'onmouseover="this.style.color=\'{palette["primary"]}\'"'
            f'onmouseout="this.style.color=\'{palette["text_secondary"]}\'">'
            f'{s.capitalize()}</a>'
            for s in sections if s != 'footer'
        ])

        sections_html = '\n'.join([
            HTMLPageBuilder.generate_section_html(s, palette)
            for s in sections
        ])

        html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport"
    content="width=device-width, initial-scale=1.0">
  <title>{title}</title>
  {css}
</head>
<body>

  <nav style="position:sticky;top:0;z-index:999;
    background:{palette['surface']}cc;
    backdrop-filter:blur(12px);
    border-bottom:1px solid {palette['border']};
    padding:18px 40px;display:flex;
    justify-content:space-between;align-items:center;">
    <span style="color:{palette['primary']};font-size:1.4rem;
      font-weight:800;">{title}</span>
    <div>{nav_links}</div>
  </nav>

  {sections_html}

</body>
</html>"""

        info = (
            f"✅ **Generated!**\n\n"
            f"**Style:** {style}\n\n"
            f"**Sections found:** {', '.join(sections)}\n\n"
            f"**Primary color:** {palette['primary']}"
        )

        return html, info

print("✅ HTMLGenerationEngine Ready!")

✅ HTMLGenerationEngine Ready!


In [7]:
with gr.Blocks(
    title="🎨 HTML Quick-Styler",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # 🎨 HTML Quick-Styler Advanced
    ### Professional HTML Page Generator with Auto-Styling
    *Generate production-ready HTML pages in seconds*
    """)

    with gr.Tabs():

        # ── TAB 1: Generate Page ──────────────────────────
        with gr.TabItem("✨ Generate Page"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 📋 Page Details")
                    page_title = gr.Textbox(
                        label="Page Title",
                        value="Modern Portfolio Website",
                        lines=1
                    )
                    page_description = gr.Textbox(
                        label="Page Description & Sections",
                        value="Hero section with welcome message, "
                              "features list with 3 items, "
                              "about section, portfolio gallery, "
                              "testimonials, contact form, footer",
                        lines=4
                    )
                    color_style = gr.Dropdown(
                        choices=["modern","vibrant","ocean",
                                 "forest","sunset","luxury",
                                 "creative","minimal"],
                        value="modern",
                        label="🎨 Color Style"
                    )
                    gen_btn = gr.Button(
                        "✨ Generate HTML",
                        variant="primary",
                        size="lg"
                    )

                with gr.Column(scale=1):
                    gr.Markdown("### 💻 Output")
                    html_output = gr.Code(
                        label="Generated HTML/CSS",
                        language="html",
                        lines=20
                    )
                    info_output = gr.Markdown()

            gen_btn.click(
                fn=HTMLGenerationEngine.generate_page,
                inputs=[page_title, page_description, color_style],
                outputs=[html_output, info_output]
            )

        # ── TAB 2: Live Preview ───────────────────────────
        with gr.TabItem("👁️ Live Preview"):
            gr.Markdown("### 🖥️ Preview Generated Page")
            load_btn = gr.Button(
                "▶ Load Preview",
                variant="primary",
                size="lg"
            )
            preview = gr.HTML(
                value="<div style='text-align:center;"
                      "padding:60px;color:#888;'>"
                      "Generate a page first, then click "
                      "Load Preview</div>"
            )

            def show_preview(code):
                if not code or not code.strip():
                    return ("<div style='text-align:center;"
                            "padding:60px;color:#888;'>"
                            "No HTML yet!</div>")
                return code

            load_btn.click(
                fn=show_preview,
                inputs=[html_output],
                outputs=[preview]
            )

        # ── TAB 3: Color Palettes ─────────────────────────
        with gr.TabItem("🌈 Color Palettes"):
            gr.Markdown("### 🎨 Explore Color Schemes")
            with gr.Row():
                style_radio = gr.Radio(
                    choices=["modern","vibrant","ocean",
                             "forest","sunset","luxury",
                             "creative","minimal"],
                    value="modern",
                    label="Choose Style"
                )
                pal_btn = gr.Button(
                    "🎨 Get Palette",
                    variant="primary"
                )
            pal_out = gr.Markdown()

            def show_palette(style):
                p = ColorPaletteGenerator.generate_palette(style)
                return (
                    f"### Palette: **{style.upper()}**\n\n"
                    f"| Property | Color |\n|---|---|\n" +
                    "\n".join([
                        f"| {k.replace('_',' ').title()} "
                        f"| `{v}` |"
                        for k,v in p.items()
                    ])
                )

            pal_btn.click(
                fn=show_palette,
                inputs=[style_radio],
                outputs=[pal_out]
            )

        # ── TAB 4: Templates ──────────────────────────────
        with gr.TabItem("📄 Templates"):
            gr.Markdown("### 📋 Quick Start Templates")
            with gr.Row():
                with gr.Column():
                    gr.Markdown("#### 🖼️ Portfolio Website\n"
                                "Hero, Gallery, About, Contact")
                    btn1 = gr.Button("Use This Template")

                with gr.Column():
                    gr.Markdown("#### 🚀 SaaS Landing Page\n"
                                "Hero, Features, Pricing, CTA")
                    btn2 = gr.Button("Use This Template")

                with gr.Column():
                    gr.Markdown("#### 🏢 Corporate Website\n"
                                "Header, About, Services, Team")
                    btn3 = gr.Button("Use This Template")

            btn1.click(
                fn=lambda: (
                    "My Portfolio",
                    "hero section, portfolio gallery, "
                    "about section, contact form, footer",
                    "modern"
                ),
                outputs=[page_title, page_description,
                         color_style]
            )
            btn2.click(
                fn=lambda: (
                    "My SaaS Product",
                    "hero banner, features showcase, "
                    "pricing plans, testimonials, "
                    "contact form, footer",
                    "vibrant"
                ),
                outputs=[page_title, page_description,
                         color_style]
            )
            btn3.click(
                fn=lambda: (
                    "Our Company",
                    "hero section, about company, "
                    "services, team, contact, footer",
                    "minimal"
                ),
                outputs=[page_title, page_description,
                         color_style]
            )

        # ── TAB 5: Guide ──────────────────────────────────
        with gr.TabItem("📖 Guide"):
            gr.Markdown("""
# 📖 HTML Quick-Styler Guide

## How It Works
1. **Enter Details** — Title + Description + Color Style
2. **Click Generate** — AI detects sections automatically
3. **Preview** — See rendered page in Live Preview tab

## Section Keywords
| Type | Keywords to use |
|------|----------------|
| Hero | hero, banner, landing |
| Features | features, services, benefits |
| About | about, team, company |
| Portfolio | portfolio, gallery, projects |
| Testimonials | testimonials, reviews |
| Pricing | pricing, plans, cost |
| Contact | contact, form, reach |
| Footer | footer, bottom |

## 8 Color Styles
`modern` `vibrant` `ocean` `forest`
`sunset` `luxury` `creative` `minimal`
            """)

# ─── LAUNCH ──────────────────────────────────────────────
print("\n✅ HTML Quick-Styler Ready!\n")
demo.launch(share=True, show_error=True)

/tmp/ipykernel_456/690414580.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(



✅ HTML Quick-Styler Ready!

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed7b2eeabb895711fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
